In [1]:
# 구글 코랩(Colab)에서 바로 실행 가능 — pandas, matplotlib 기본 설치됨
import urllib.request   # URL로 요청을 보내는 기본 모듈
import urllib.parse     # 한글 검색어를 URL용으로 인코딩
import json             # JSON 문자열 ↔ 파이썬 딕셔너리 변환
import datetime         # 시간 기록

In [2]:
import urllib.request

res = urllib.request.urlopen("https://www.naver.com")
print(type(res))     # <class 'http.client.HTTPResponse'>
print(res.status)    # 200  → 요청 성공
print("--- 헤더 정보 ---")
for h in res.getheaders():
    print(h)

<class 'http.client.HTTPResponse'>
200
--- 헤더 정보 ---
('Date', 'Thu, 04 Jun 2026 02:20:45 GMT')
('content-type', 'text/html; charset=UTF-8')
('transfer-encoding', 'chunked')
('set-cookie', 'PM_CK_loc=6a99d2fc1b5471d9d14cea36ec9cb8ccbc593048dad883183ee247e5610d9706; Expires=Fri, 05 Jun 2026 02:20:45 GMT; Path=/; HttpOnly')
('cache-control', 'no-cache, no-store, must-revalidate')
('pragma', 'no-cache')
('x-frame-options', 'DENY')
('strict-transport-security', 'max-age=63072000; includeSubdomains')
('referrer-policy', 'unsafe-url')
('server', 'nfront')
('connection', 'close')


In [3]:
import urllib.request
import urllib.parse

# 1) https://developers.naver.com 에서 애플리케이션 등록 후 발급
client_id = "oKzSKXj9qj4h10oQ55Py"
client_secret = "Ud2M4S4BXB"

encText = urllib.parse.quote("애플")   # 한글 검색어 → URL 인코딩 필수
url = "https://openapi.naver.com/v1/search/blog?query=" + encText

request = urllib.request.Request(url)
request.add_header("X-Naver-Client-Id", client_id)        # 인증키를 헤더에 실어 보냄
request.add_header("X-Naver-Client-Secret", client_secret)

response = urllib.request.urlopen(request)
if response.getcode() == 200:
    body = response.read().decode("utf-8")   # 바이트 → 문자열
    print(body)
else:
    print("Error Code:", response.getcode())

{
	"lastBuildDate":"Thu, 04 Jun 2026 11:35:09 +0900",
	"total":8165706,
	"start":1,
	"display":10,
	"items":[
		{
			"title":"뉴욕 여행 준비는 빅<b>애플<\/b>패스 뉴욕 3대 패스 비교 및 추천",
			"link":"https:\/\/blog.naver.com\/be_ok-\/224294982259",
			"description":"뉴욕 여행 준비는 빅<b>애플<\/b>패스 뉴욕 3대 패스 비교 및 추천 안녕하세요! 여행 인플루언서 홍지구입니다... 지구가 찾아봤던 뉴욕 3대 패스를 한 번 비교해보고 지구가 선택한 빅<b>애플<\/b>패스가 어떤 장점이... ",
			"bloggername":"오늘도, 지구 여행",
			"bloggerlink":"blog.naver.com\/be_ok-",
			"postdate":"20260524"
		},
		{
			"title":"아오모리 골프여행 | <b>애플<\/b>랜드 온천호텔 투숙 후기",
			"link":"https:\/\/blog.naver.com\/trip_pro\/224298640835",
			"description":"이번에 소개해 드릴 곳은 아오모리 골프여행에서 한국인들에게 가장 인기 있는 온천호텔 <b>애플<\/b>랜드입니다. 많은 분들이 궁금해합니다. 왜 아오모리로 골프여행을 가면 시내 호텔이 아닌 <b>애플<\/b>랜드로만 가느냐고... ",
			"bloggername":"바로여행 최팀장의 골프여행",
			"bloggerlink":"blog.naver.com\/trip_pro",
			"postdate":"20260528"
		},
		{
			"title":"<b>애플<\/b>워치 울트라3 추천 리뷰 명품 스포츠 아웃도어 스마트워치",
			"link":"https:\/\/blog.naver.com\/happypanchok\/224301214586",
			"description

In [1]:
import os, urllib.request, urllib.parse, datetime, json

client_id = "oKzSKXj9qj4h10oQ55Py"
client_secret = "Ud2M4S4BXB"

def getRequestUrl(url):                       # [CODE 1] 공통 요청 함수
    req = urllib.request.Request(url)
    req.add_header("X-Naver-Client-Id", client_id)
    req.add_header("X-Naver-Client-Secret", client_secret)
    try:
        response = urllib.request.urlopen(req)
        if response.getcode() == 200:
            print("[%s] Url Request Success" % datetime.datetime.now())
            return response.read().decode("utf-8")
    except Exception as e:
        print(e); return None

def getNaverSearch(node, srcText, start, display):   # [CODE 2] 검색 요청 조립
    base = "https://openapi.naver.com/v1/search"
    node = "/%s.json" % node
    params = "?query=%s&start=%s&display=%s" % (urllib.parse.quote(srcText), start, display)
    responseDecode = getRequestUrl(base + node + params)
    return None if responseDecode is None else json.loads(responseDecode)

def getPostData(post, jsonResult, cnt):              # [CODE 3] 필요한 항목만 추출
    pDate = datetime.datetime.strptime(post['postdate'], '%Y%m%d').strftime('%Y-%m-%d')
    jsonResult.append({'cnt': cnt, 'title': post['title'],
                       'description': post['description'],
                       'link': post['bloggerlink'], 'pDate': pDate})

def main():
    node = 'blog'
    srcText = input('검색어를 입력하세요: ')
    cnt = 0; jsonResult = []
    jsonResponse = getNaverSearch(node, srcText, 1, 100)
    total = jsonResponse['total']
    while jsonResponse is not None and jsonResponse['display'] != 0:
        for post in jsonResponse['items']:
            cnt += 1
            getPostData(post, jsonResult, cnt)
        start = jsonResponse['start'] + jsonResponse['display']
        jsonResponse = getNaverSearch(node, srcText, start, 100)
    print('전체 검색: %d 건 / 수집: %d 건' % (total, cnt))
    with open('%s_naver_%s.json' % (srcText, node), 'w', encoding='utf8') as f:
        f.write(json.dumps(jsonResult, indent=4, ensure_ascii=False))
    print('저장 완료')

if __name__ == '__main__':
    main()

[2026-06-05 09:23:06.978740] Url Request Success
[2026-06-05 09:23:07.332626] Url Request Success
[2026-06-05 09:23:07.677222] Url Request Success
[2026-06-05 09:23:08.017351] Url Request Success
[2026-06-05 09:23:08.365587] Url Request Success
[2026-06-05 09:23:08.759545] Url Request Success
[2026-06-05 09:23:09.114198] Url Request Success
[2026-06-05 09:23:09.477287] Url Request Success
[2026-06-05 09:23:09.854042] Url Request Success
[2026-06-05 09:23:10.209577] Url Request Success
HTTP Error 400: Bad Request
전체 검색: 33801741 건 / 수집: 1000 건
저장 완료


In [5]:
import urllib.request, urllib.parse, json, ssl, platform, datetime
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

API_KEY = "7e3732823bcfddf911e567ea22fe8bfcb55a3cc9e0ce0d9512bcf6c9d0312123"   # data.go.kr 에서 발급 (Decoding 키 사용)

SEOUL_DISTRICTS = {"종로구": 110, "중구": 140, "용산구": 170, "성동구": 200,
                   "강남구": 680, "서초구": 650, "송파구": 710}   # 일부 예시

context = ssl.create_default_context()
context.set_ciphers('DEFAULT@SECLEVEL=1')   # 일부 공공기관 구형 SSL 대응

def set_korean_font():                       # 한글 폰트 (OS 자동 감지)
    sysname = platform.system()
    path = {"Windows": "C:/Windows/Fonts/malgun.ttf",
            "Darwin": "/System/Library/Fonts/Supplemental/AppleGothic.ttf"}\
           .get(sysname, "/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    plt.rcParams["font.family"] = fm.FontProperties(fname=path).get_name()
    plt.rcParams["axes.unicode_minus"] = False
set_korean_font()

def fetch(searchYearCd, siDo, guGun, numOfRows=100, pageNo=1):
    base = "https://apis.data.go.kr/B552061/frequentzoneBicycle/getRestFrequentzoneBicycle"
    params = {"ServiceKey": API_KEY, "searchYearCd": searchYearCd, "siDo": siDo,
              "guGun": guGun, "type": "json", "numOfRows": numOfRows, "pageNo": pageNo}
    url = base + "?" + urllib.parse.urlencode(params)   # 파라미터 자동 인코딩
    req = urllib.request.Request(url)
    try:
        text = urllib.request.urlopen(req, context=context).read().decode("utf-8")
    except Exception as e:
        print("오류:", e); return None
    if not text.strip():
        print("응답이 비어있음"); return None
    return json.loads(text)

# 강남구(680) 2023년 사고 다발 지역 수집 예시
data = fetch("2023", "11", "680")
if data and data.get("items"):
    df = pd.DataFrame(data["items"]["item"])
    print(df.head())
    df["occrrnc_cnt"] = pd.to_numeric(df["occrrnc_cnt"])  # 발생건수 숫자화
    plt.figure(figsize=(10, 5))
    plt.bar(df["spot_nm"], df["occrrnc_cnt"], color="skyblue")
    plt.title("2023 서울 강남구 자전거 사고 다발지역")
    plt.xticks(rotation=45, ha="right"); plt.tight_layout()
    plt.show()

오류: HTTP Error 403: Forbidden
